# Credit Card Fraud Detection using SMOTE

This notebook implements end-to-end Machine Learning and Deep Learning (ANN) pipelines to classify highly imbalanced Credit Card Fraud transactions with and without **SMOTE** (Synthetic Minority Over-sampling Technique).

### 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, classification_report, confusion_matrix, roc_curve, auc

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

### 2. Download and Load OpenML Dataset

In [ ]:
print("Download the datasets from internet")
# OpenML 'CreditCardFraudDetection' (ID: 1597) dataset
credit_data = fetch_openml(data_id=1597, as_frame=True, parser='auto')
print("Download complete")

In [ ]:
df = credit_data.frame
print(f"data shape (Rows, Columns): {df.shape}")

In [ ]:
df.head()

In [ ]:
df.info()

### 3. Check Class Imbalance & Preprocess Data

In [ ]:
df['Class'] = df['Class'].astype(int)
print("Class distribution in data:\n", df['Class'].value_counts())

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 4. Apply SMOTE to Balance Training Data

In [ ]:
print("\n--- SMOTE ---")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print("after SMOT size of data:", pd.Series(y_train_smote).value_counts())

### 5. Train Machine Learning Models on SMOTE Data

In [ ]:
real_model_names = []
real_accuracy_scores = []
real_recall_scores = []

ml_models = {
    'Random Forest': RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1),
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'SVM': LinearSVC(random_state=42, dual=False, max_iter=2000),
    'Naive Bayes': GaussianNB()
}

In [ ]:
print("\n--- 3. Training of machine learnings models... ---")
for name, model in ml_models.items():
    print(f" Training: {name}...")
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test_scaled)
    
    real_model_names.append(name)
    real_accuracy_scores.append(accuracy_score(y_test, y_pred) * 100)
    real_recall_scores.append(recall_score(y_test, y_pred) * 100)

### 6. Deep Learning: Artificial Neural Network (ANN)

In [ ]:
print("\n--- 4. (Neural Networ) Training... ---")
ann_real = Sequential([
    Dense(32, activation='relu', input_shape=(X_train_smote.shape[1],)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

ann_real.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
ann_real.fit(X_train_smote, y_train_smote, epochs=5, batch_size=128, verbose=0)

ann_prob = ann_real.predict(X_test_scaled).flatten()
y_pred_ann = np.where(ann_prob > 0.5, 1, 0)

real_model_names.append('Neural Network')
real_accuracy_scores.append(accuracy_score(y_test, y_pred_ann) * 100)
real_recall_scores.append(recall_score(y_test, y_pred_ann) * 100)

print("\n--- models graph... ---")

### 7. Performance Visualization: Accuracy vs Recall Comparison

In [ ]:
df_real_compare = pd.DataFrame({
    'Model': real_model_names * 2,
    'Percentage (%)': real_accuracy_scores + real_recall_scores,
    'Metric': ['Accuracy'] * len(real_model_names) + ['Recall (Fraud Rate)'] * len(real_model_names)
})

plt.figure(figsize=(14, 7))
ax = sns.barplot(data=df_real_compare, x='Model', y='Percentage (%)', hue='Metric', palette='Set2')

for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{p.get_height():.1f}%",
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center',
                    xytext=(0, 8),
                    textcoords='offset points',
                    fontsize=10, fontweight='bold')

plt.title('Final performance', fontsize=16, fontweight='bold')
plt.xlabel('ML and DL Algo', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.ylim(0, 115)
plt.legend(title='Metrics')
plt.tight_layout()
plt.show()

# With and Without SMOTE Comparison

### 8. Classification Report: Without SMOTE vs With SMOTE

In [ ]:
for name, model in ml_models.items():
    print("=" * 60)
    print(f" Algorithms: {name} ".center(60, "="))
    print("=" * 60)
    
    # Training (Without SMOTE)
    print(f"\n--- [1] {name} (without smote) ---")
    model.fit(X_train_scaled, y_train)
    y_pred_no_smote = model.predict(X_test_scaled)
    print(classification_report(y_test, y_pred_no_smote))
    
    # Training (With SMOTE)
    print(f"--- [2] {name} (with smote) ---")
    model.fit(X_train_smote, y_train_smote)
    y_pred_with_smote = model.predict(X_test_scaled)
    print(classification_report(y_test, y_pred_with_smote))

### 9. Confusion Matrix Heatmaps: With and Without SMOTE

In [ ]:
# ==========================================================
# Confusion Matrix with and Without Smote
# ==========================================================
ml_models = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1),
    'SVM (Linear SVC)': LinearSVC(random_state=42, dual=False, max_iter=2000),
    'Naive Bayes': GaussianNB()
}

# ----------------------------------------------------------
# Models Loop
# ----------------------------------------------------------
for name, model in ml_models.items():
    # 1. Without SMOTE
    model.fit(X_train_scaled, y_train)
    y_pred_no_smote = model.predict(X_test_scaled)
    cm_no_smote = confusion_matrix(y_test, y_pred_no_smote)
    
    # 2. With SMOTE
    model.fit(X_train_smote, y_train_smote)
    y_pred_with_smote = model.predict(X_test_scaled)
    cm_with_smote = confusion_matrix(y_test, y_pred_with_smote)
    
    # Set the graph shapes
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Graph without SMOTE
    sns.heatmap(cm_no_smote, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
                xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'],
                annot_kws={"size": 12, "weight": "bold"})
    axes[0].set_title(f'{name} (Without SMOTE)', fontsize=13, fontweight='bold', color='navy')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('Actual')
    
    # Graph with SMOTE
    sns.heatmap(cm_with_smote, annot=True, fmt='d', cmap='Greens', cbar=False, ax=axes[1],
                xticklabels=['Normal', 'Fraud'], yticklabels=['Normal', 'Fraud'],
                annot_kws={"size": 12, "weight": "bold"})
    axes[1].set_title(f'{name} (With SMOTE)', fontsize=13, fontweight='bold', color='darkgreen')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    
    plt.suptitle(f'== Confusion Matrix: {name} ==', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()